# EPUB Audiobook - batch chunk synthesis (multiple patches, one run)

This notebook synthesizes the text chunks exported by the EPUB Audiobook App
for **every patch in the batch**, sequentially, using the same `VoxCPM2` model
the app uses locally. For each patch it writes `chunk_NNN.wav` files into an
`output/` subfolder inside that patch's folder, and as soon as a patch is
complete it merges the chunks into a single **`result/NNN - <patch name>.wav`**
at the batch root.

> **Set `IS_KAGGLE` first.** Cell 1 defines one global flag, `IS_KAGGLE`, that
> every cell in this notebook uses — there is **no per-cell auto-detection**.
> Set it to `True` when running on Kaggle or `False` when running on Google
> Colab, then run the cells top to bottom.

> **Enable a GPU first.** This model runs on CUDA. In Colab: **Runtime > Change runtime type > GPU (T4)**, then restart the session. Pick **GPU, not TPU** - VoxCPM cannot use a TPU and will silently fall back to CPU (extremely slow). Cell 6 checks this for you.

It reads everything it needs from `batch_manifest.json`, which was exported
alongside this notebook - you should not need to type any patch info by hand.

## No re-downloading the model on restart
Cell 1 points the Hugging Face cache at persistent storage, so the multi-GB
`VoxCPM2` weights are downloaded **once**, not on every session:

- **Colab**: cached in your Drive at `EPUB Audiobook Exports/.cache` (the
  first run is a normal download that lands in Drive; later sessions load
  from there). Uses a few GB of Drive space.
- **Kaggle**: cached in `/kaggle/working/.cache` - turn on
  **Persistence: Files only** in the notebook options so it survives new
  sessions.

## Disconnects are fine - just re-run
Free Colab/Kaggle sessions can die mid-run. The synthesis cell is safe to
re-run any number of times:

- chunks that already have a `.wav` are **skipped**,
- patches that already have their merged `result/` file are **skipped entirely**,
- on Colab every `.wav` is written **directly into your Drive folder**, so
  progress survives even if the runtime is killed - reconnect, run the cells
  top to bottom (the model reloads), and it continues where it stopped.

## Google Colab (recommended)
Keep `IS_KAGGLE = False` in Cell 1. The app already uploaded this folder into
**your own Google Drive** (the account you connected). Just run the cells top
to bottom: cell 3 mounts your Drive, and the folder is a normal filesystem
path from then on - no Google API calls needed in this notebook at all. Merged
patch files end up in the `result/` subfolder of the batch folder in Drive.

## Kaggle (recommended: Drive via Kaggle Secret)
Set `IS_KAGGLE = True` in Cell 1. Kaggle has no native Google Drive mount, but
Cell 4 talks to the Drive API directly using the app's own credentials, giving
Kaggle the same experience as Colab. One-time setup:

1. In the app, open the **Google Drive** page and click
   **Copy Kaggle credentials** (requires Drive to be connected).
2. On Kaggle: **Add-ons > Secrets** > add a secret named **`GDRIVE_CREDS`**
   with that JSON as the value, and enable it for this notebook.

Cell 4 then downloads the exported batch folder from Drive into
`/kaggle/working` (including any `.wav` chunks from earlier sessions), and
Cell 8 uploads every generated `.wav` and merged `result/` file **straight
back to Drive** as it is written. A dead Kaggle session resumes exactly like
Colab - re-run the cells and it continues - and the app can import the
results from Drive as usual.

### Kaggle without Drive (zip-dataset fallback)
If you'd rather not store credentials on Kaggle (still set `IS_KAGGLE = True`):
1. In the app, use **Download selected (.zip)**, upload the zip as a Kaggle
   Dataset and attach it to this notebook.
2. Skip Cells 3 and 4 and set `FOLDER_PATH` to the dataset path
   (e.g. `/kaggle/input/<dataset-name>`) - see the comment in Cell 4.
3. Output goes under `/kaggle/working`; run the last cell to zip `result/`
   and download it from Kaggle's **Output** pane.
4. **Resuming across sessions:** `/kaggle/working` does not survive a new
   session. To resume, add the `.wav` files you already produced to the
   dataset under `patches/patch_NNN/output/` - the skip check looks there
   too and will not redo those chunks.

## Video rendering (Cell 10–11)
After TTS synthesis is complete, **Cell 10** checks FFmpeg is available
(pre-installed on Colab/Kaggle). **Cell 11** then renders one MP4 per patch:
- Uses the background image bundled in the package (already has book title +
  patch name baked in by the app — no font needed here).
- Mixes the TTS audio with optional background music from `music/` (loop,
  low volume), if the book had music assigned when exported.
- Saves each MP4 to `result/NNN - <patch name>.mp4`.
- Skip-safe: if the MP4 already exists it is not re-rendered.

## YouTube upload (Cell 12)
**Cell 12** uploads every rendered MP4 directly to YouTube using a
`YOUTUBE_CREDS` Kaggle/Colab Secret. To set it up:
1. In the app, open **/youtube** and click **Copy YouTube credentials for Kaggle/Colab**.
2. On Kaggle: **Add-ons > Secrets** > add secret `YOUTUBE_CREDS` with that JSON.
   On Colab: left sidebar key icon > add secret `YOUTUBE_CREDS`.
- Skip-safe: if a `result/NNN.mp4.youtube_id` file exists, that video is skipped.

In [ ]:
# Cell 1: platform flag + persistent caches + dependencies.
#
# >>> SET THIS FIRST <<<
# IS_KAGGLE is the single global switch used by EVERY cell in this notebook -
# there is no per-cell auto-detection. Set it before running anything:
#   True  -> running on Kaggle
#   False -> running on Google Colab
IS_KAGGLE = False

# Persistent caches: point the Hugging Face model cache (and pip's download
# cache) at persistent storage, so restarting the session does NOT re-download
# the multi-GB model weights every time:
#   - Colab: cached in your Google Drive under "EPUB Audiobook Exports/.cache".
#     The first run downloads the model once into Drive; every later session
#     loads it from there.
#   - Kaggle: cached in /kaggle/working/.cache. Enable the notebook's
#     "Persistence: Files only" setting (right sidebar > Notebook options) so
#     the cache survives across sessions - without it the cache still helps
#     within one session, but a brand-new session starts empty.
import os

USE_PERSISTENT_CACHE = True  # set False to use the default ephemeral cache

CACHE_ROOT = None
if USE_PERSISTENT_CACHE:
    if IS_KAGGLE:
        CACHE_ROOT = "/kaggle/working/.cache"
    else:
        try:
            from google.colab import drive
            drive.mount('/content/drive')  # also mounted by Cell 3; mounting twice is fine
            CACHE_ROOT = "/content/drive/MyDrive/EPUB Audiobook Exports/.cache"
        except Exception as exc:
            print(f"Drive not available ({type(exc).__name__}) - using ephemeral cache.")

if CACHE_ROOT:
    os.makedirs(CACHE_ROOT, exist_ok=True)
    # Must be set BEFORE anything imports huggingface_hub (it reads HF_HOME at
    # import time), which is why this cell comes first.
    os.environ["HF_HOME"] = os.path.join(CACHE_ROOT, "huggingface")
    # Drive's FUSE mount doesn't support symlinks; the HF cache detects that and
    # falls back to plain file copies - silence the warning about it.
    os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
    os.environ["PIP_CACHE_DIR"] = os.path.join(CACHE_ROOT, "pip")
    print("Persistent cache:", CACHE_ROOT)
else:
    print("No persistent cache - the model will be re-downloaded each session.")

!pip install -q voxcpm soundfile numpy

In [ ]:
# Cell 2: (optional) Hugging Face token - avoids the "unauthenticated requests" rate
# limit warning/slow downloads when fetching the model. Get a free token at
# https://huggingface.co/settings/tokens
#
# Recommended: store it as a secret instead of pasting it in plain text here -
# Colab: left sidebar > key icon > add secret named HF_TOKEN.
# Kaggle: Add-ons > Secrets > add secret named HF_TOKEN.
# If no secret is found, you'll get a hidden prompt to paste it manually (or just
# press Enter to skip and continue unauthenticated).
# The app replaces __HF_TOKEN__ below with the token from its own settings on export;
# leave the app's HF_TOKEN setting empty to keep this as a placeholder (secrets/prompt).
# Which secret store is read follows the global IS_KAGGLE flag from Cell 1.
HF_TOKEN = "__HF_TOKEN__"

if not HF_TOKEN:
    if IS_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN") or ""
        except Exception:
            pass
    else:
        try:
            from google.colab import userdata
            HF_TOKEN = userdata.get("HF_TOKEN") or ""
        except Exception:
            pass

if not HF_TOKEN:
    import getpass
    HF_TOKEN = getpass.getpass("Hugging Face token (leave blank to skip): ")

if HF_TOKEN:
    import os
    os.environ["HF_TOKEN"] = HF_TOKEN
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("Logged in to Hugging Face Hub.")
else:
    print("No HF token set - continuing unauthenticated (may hit rate limits).")

In [ ]:
# Cell 3: Google Colab only - mount your Drive. The exported batch folder is located
# automatically by batch id, so you do NOT need to paste the folder name by hand.
# Uses the global IS_KAGGLE flag from Cell 1: on Kaggle this cell does nothing
# (Cell 4 takes over), so "Run all" works on both platforms.
import glob, json, os

if IS_KAGGLE:
    print("IS_KAGGLE = True - skipping the Colab Drive mount (Cell 4 downloads the batch instead).")
else:
    from google.colab import drive

    drive.mount('/content/drive')

    BATCH_ID = "__BATCH_ID__"  # injected by the app when this notebook was exported
    EXPORTS_ROOT = "/content/drive/MyDrive/EPUB Audiobook Exports"
    DEFAULT_FOLDER = os.path.join(EXPORTS_ROOT, "__DEFAULT_FOLDER_NAME__")

    def _read_batch_id(folder):
        """Return the batch_id inside folder/batch_manifest.json, or a short reason string."""
        manifest_path = os.path.join(folder, "batch_manifest.json")
        if not os.path.isfile(manifest_path):
            return "<no batch_manifest.json>"
        try:
            with open(manifest_path, encoding="utf-8") as f:
                return json.load(f).get("batch_id")
        except Exception as exc:
            return f"<unreadable batch_manifest.json: {exc}>"

    # Scan every export folder and match the one whose batch_manifest.json has our batch id.
    FOLDER_PATH = None
    for d in sorted(glob.glob(os.path.join(EXPORTS_ROOT, "*")), reverse=True):
        if os.path.isdir(d) and _read_batch_id(d) == BATCH_ID:
            FOLDER_PATH = d
            break

    if FOLDER_PATH is None:
        FOLDER_PATH = DEFAULT_FOLDER  # fall back to the exact name the app used at export time

    print("Using folder:", FOLDER_PATH)
    if not os.path.isdir(FOLDER_PATH):
        # Folder check: list what actually is under the exports root so a wrong Google
        # account or an incomplete rclone push is obvious at a glance.
        print(f"\nbatch_id {BATCH_ID} not found. Export folders under {EXPORTS_ROOT}:")
        _visible = [d for d in sorted(glob.glob(os.path.join(EXPORTS_ROOT, "*"))) if os.path.isdir(d)]
        if _visible:
            for d in _visible:
                print(f"  - {os.path.basename(d)}  (batch_id={_read_batch_id(d)})")
        else:
            print(f"  (nothing - {EXPORTS_ROOT} is empty or missing on the account you mounted)")
        print(
            "\nIf the folder is missing above: you mounted a different Google account than the "
            "one the batch was exported to, or the rclone push has not finished. Mount the "
            "account that owns the batch, or set FOLDER_PATH manually to one of the folders listed."
        )
    assert os.path.isdir(FOLDER_PATH), f"Folder not found: {FOLDER_PATH}"


In [ ]:
# Cell 4: Kaggle only - connect to Google Drive with the GDRIVE_CREDS secret and
# download the exported batch folder. Uses the global IS_KAGGLE flag from Cell 1:
# on Colab this cell does nothing (Cell 3 already mounted Drive), so "Run all"
# works on both platforms.
#
# One-time setup (see the intro above): in the app open the Google Drive page and
# click "Copy Kaggle credentials", then on Kaggle add it via Add-ons > Secrets as a
# secret named GDRIVE_CREDS and enable it for this notebook.
#
# This cell downloads the batch (including output/ wavs from earlier sessions) into
# /kaggle/working/batch, and defines drive_persist() which Cell 8 uses to upload
# every generated .wav and merged result file straight back to Drive.
#
# --- zip-dataset fallback (no Drive credentials) ---
# Skip this cell too and point FOLDER_PATH at the attached dataset instead. Use the
# EXACT zip filename (without .zip) as the dataset name when uploading, e.g.:
# FOLDER_PATH = "/kaggle/input/<dataset-name>"
import io
import json
import os

if not IS_KAGGLE:
    print("IS_KAGGLE = False - skipping the Kaggle Drive download (Cell 3 already mounted Drive).")
else:
    !pip install -q google-api-python-client google-auth

    from google.oauth2.credentials import Credentials
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload
    from kaggle_secrets import UserSecretsClient

    BATCH_ID = "__BATCH_ID__"  # injected by the app when this notebook was exported

    creds_info = json.loads(UserSecretsClient().get_secret("GDRIVE_CREDS"))
    creds = Credentials(
        token=None,
        refresh_token=creds_info["refresh_token"],
        token_uri="https://oauth2.googleapis.com/token",
        client_id=creds_info["client_id"],
        client_secret=creds_info["client_secret"],
        scopes=["https://www.googleapis.com/auth/drive.file"],
    )
    drive_service = build("drive", "v3", credentials=creds)

    FOLDER_MIME = "application/vnd.google-apps.folder"


    def _list_children(folder_id):
        files, token = [], None
        while True:
            resp = drive_service.files().list(
                q=f"'{folder_id}' in parents and trashed = false",
                fields="nextPageToken, files(id, name, mimeType)",
                pageToken=token, pageSize=1000,
            ).execute()
            files += resp.get("files", [])
            token = resp.get("nextPageToken")
            if not token:
                return files


    def _download(file_id, dest):
        request = drive_service.files().get_media(fileId=file_id)
        downloader = MediaIoBaseDownload(dest, request)
        done = False
        while not done:
            _, done = downloader.next_chunk()


    # Locate the batch folder: scan "EPUB Audiobook Exports" for the folder whose
    # batch_manifest.json carries this notebook's batch id.
    resp = drive_service.files().list(
        q=f"name = 'EPUB Audiobook Exports' and mimeType = '{FOLDER_MIME}' and trashed = false",
        fields="files(id)",
    ).execute()
    _roots = resp.get("files", [])
    assert _roots, (
        "No 'EPUB Audiobook Exports' folder found. The credentials only see files "
        "created by the app itself - make sure you exported this batch to Drive."
    )

    _batch_folder_id = None
    _seen_folders = []  # (name, batch_id) for every export folder these creds can see
    for _root in _roots:
        for _folder in _list_children(_root["id"]):
            if _folder["mimeType"] != FOLDER_MIME:
                continue
            _mf = next((f for f in _list_children(_folder["id"]) if f["name"] == "batch_manifest.json"), None)
            if _mf is None:
                _seen_folders.append((_folder["name"], "<no batch_manifest.json>"))
                continue
            _buf = io.BytesIO()
            _download(_mf["id"], _buf)
            _found_id = json.loads(_buf.getvalue().decode("utf-8")).get("batch_id")
            _seen_folders.append((_folder["name"], _found_id))
            if _found_id == BATCH_ID:
                _batch_folder_id = _folder["id"]
                print("Found batch folder on Drive:", _folder["name"])
                break
        if _batch_folder_id:
            break

    if not _batch_folder_id:
        # Folder check: show exactly what these credentials CAN see, so a mismatch is
        # obvious. The drive.file scope only reveals folders THIS OAuth client created,
        # so a batch pushed via rclone or exported to a different Google account is
        # invisible here even though it exists on Drive.
        print(f"\nbatch_id {BATCH_ID} not found. Export folders these credentials can see:")
        if _seen_folders:
            for _name, _bid in _seen_folders:
                print(f"  - {_name}  (batch_id={_bid})")
        else:
            print("  (none - this account has no app-created export folders)")
        raise AssertionError(
            f"No Drive folder found with batch_id {BATCH_ID}. If it is missing above, the "
            "batch was pushed via rclone or to a different Google account (the drive.file "
            "scope cannot see either). Use the account that owns the batch, or fall back to "
            "the zip-dataset method: skip this cell and set FOLDER_PATH = /kaggle/input/<dataset-name>."
        )

    # Download the whole batch folder - including output/ wavs from earlier sessions,
    # which the skip and merge logic in Cell 8 needs locally - and remember the Drive
    # ids of every folder/file so drive_persist() can upload without re-listing.
    FOLDER_PATH = "/kaggle/working/batch"
    _drive_folder_ids = {"": _batch_folder_id}
    _drive_file_ids = {}


    def _sync_down(folder_id, rel):
        for f in _list_children(folder_id):
            child_rel = f"{rel}/{f['name']}" if rel else f["name"]
            if f["mimeType"] == FOLDER_MIME:
                _drive_folder_ids[child_rel] = f["id"]
                _sync_down(f["id"], child_rel)
            else:
                _drive_file_ids[child_rel] = f["id"]
                local = os.path.join(FOLDER_PATH, child_rel)
                if not os.path.exists(local):
                    os.makedirs(os.path.dirname(local), exist_ok=True)
                    with open(local, "wb") as fh:
                        _download(f["id"], fh)
                    print("downloaded", child_rel)


    _sync_down(_batch_folder_id, "")
    print("Batch ready at", FOLDER_PATH)


    def drive_persist(local_path, rel_dir):
        """Upload a freshly written file into rel_dir inside the batch folder on Drive,
        creating the subfolder chain as needed and replacing an existing file in place.
        Cell 8 calls this after every chunk .wav and every merged result file, so a dead
        Kaggle session loses nothing."""
        parent, rel = _drive_folder_ids[""], ""
        for part in [p for p in rel_dir.split("/") if p]:
            rel = f"{rel}/{part}" if rel else part
            if rel not in _drive_folder_ids:
                folder = drive_service.files().create(
                    body={"name": part, "mimeType": FOLDER_MIME, "parents": [parent]},
                    fields="id",
                ).execute()
                _drive_folder_ids[rel] = folder["id"]
            parent = _drive_folder_ids[rel]
        name = os.path.basename(local_path)
        file_rel = f"{rel}/{name}" if rel else name
        media = MediaFileUpload(local_path)
        if file_rel in _drive_file_ids:
            drive_service.files().update(fileId=_drive_file_ids[file_rel], media_body=media).execute()
        else:
            created = drive_service.files().create(
                body={"name": name, "parents": [parent]}, media_body=media, fields="id",
            ).execute()
            _drive_file_ids[file_rel] = created["id"]


In [ ]:
# Cell 5: load the batch manifest and the shared voice reference clip. The clip is
# REQUIRED: without it VoxCPM picks a different random voice per chunk/session, so
# the merged audio would not sound consistent.
import json
import os

with open(os.path.join(FOLDER_PATH, "batch_manifest.json"), "r", encoding="utf-8") as f:
    batch_manifest = json.load(f)

print(f"Batch of {batch_manifest['patch_count']} patches from book '{batch_manifest['book_title']}'")
for entry in batch_manifest["patches"]:
    print(f"  patch {entry['patch_index']:03d}: {entry['patch_name']} "
          f"(chapters {entry['chapter_start']}-{entry['chapter_end']}, {entry['chunk_count']} chunks)")

reference_wav_path = None
prompt_text = None
if batch_manifest.get("reference_wav"):
    reference_wav_path = os.path.join(FOLDER_PATH, batch_manifest["reference_wav"])
    prompt_text = batch_manifest.get("reference_transcript") or None

if not reference_wav_path or not os.path.exists(reference_wav_path):
    raise RuntimeError(
        "Voice reference clip not found in this export - it is required so every "
        "chunk is synthesized with the same voice. In the app, upload a voice "
        "reference clip for this book, then re-export the batch."
    )
print(f"Using cloned voice reference: {reference_wav_path}")

In [ ]:
# Cell 6: check you're actually on a GPU (VoxCPM is far too slow on CPU).
# If this stops with an error, go to Runtime > Change runtime type > Hardware
# accelerator > GPU (T4), then Runtime > Restart session and run again.
# NOTE: pick GPU, NOT TPU - VoxCPM uses CUDA and cannot run on a TPU.
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        "No GPU detected - VoxCPM would run on CPU and be extremely slow. "
        "Colab: Runtime > Change runtime type > GPU (T4), then Restart session. "
        "Choose GPU, not TPU. On Kaggle: enable a GPU accelerator in the sidebar."
    )

In [ ]:
# Cell 7: load the model ONCE - it is reused for every patch in the batch.
# Kept in its own cell so that after a disconnect you can re-run Cell 8 alone
# without waiting for a re-download.
from voxcpm import VoxCPM

model = VoxCPM.from_pretrained(batch_manifest.get("voxcpm_model_id", "openbmb/VoxCPM2"), load_denoiser=False)

In [ ]:
# Cell 8: synthesize every patch in order, merging each patch into result/ as soon
# as it completes. Safe to re-run after any disconnect:
#   - chunks with an existing .wav are skipped,
#   - patches with an existing merged result file are skipped entirely,
#   - on Colab everything below is written straight into the Drive-mounted folder,
#     so progress is persisted the moment each file is written.
import json
import os

import numpy as np
import soundfile as sf

# --- config ---
PATCH_IDS = None      # None = run ALL patches in the batch; or e.g. [12, 15] to restrict
SKIP_EXISTING = True  # skip chunks/patches that already have output (safe resume)

# Drive upload hook: defined by the Kaggle Drive cell (Cell 4); a no-op everywhere
# else - on Colab files are already written straight into the Drive mount, and the
# zip-dataset fallback has nowhere to upload to.
persist = globals().get("drive_persist") or (lambda local_path, rel_dir: None)

# /kaggle/input is a read-only mount, so with the zip-dataset fallback all output
# goes under /kaggle/working instead. Otherwise FOLDER_PATH is writable (the Drive
# mount on Colab, or /kaggle/working/batch in Kaggle Drive mode), so output/ and
# result/ live right inside the batch folder.
ON_KAGGLE_DATASET = FOLDER_PATH.startswith("/kaggle/input")
WORK_ROOT = "/kaggle/working" if ON_KAGGLE_DATASET else FOLDER_PATH
RESULT_DIR = os.path.join(WORK_ROOT, "result")
os.makedirs(RESULT_DIR, exist_ok=True)
print("Merged patch files will be written to:", RESULT_DIR)

sample_rate = model.tts_model.sample_rate
summary = []

for entry in sorted(batch_manifest["patches"], key=lambda e: e["patch_index"]):
    label = f"patch {entry['patch_index']:03d} ({entry['patch_name']})"
    if PATCH_IDS is not None and entry["patch_id"] not in PATCH_IDS:
        summary.append((label, "skipped (not in PATCH_IDS)"))
        continue

    patch_dir = os.path.join(FOLDER_PATH, entry["folder"])
    out_dir = os.path.join(WORK_ROOT, entry["folder"], "output")
    result_path = os.path.join(WORK_ROOT, entry["result_wav"])
    print(f"\n=== {label}: {entry['chunk_count']} chunks ===")

    if SKIP_EXISTING and os.path.exists(result_path):
        print(f"already merged -> {result_path} (skipping patch)")
        summary.append((label, "done (already merged)"))
        continue

    os.makedirs(out_dir, exist_ok=True)
    with open(os.path.join(patch_dir, "manifest.json"), "r", encoding="utf-8") as f:
        manifest = json.load(f)
    if "expected_outputs" not in manifest:
        manifest["expected_outputs"] = [f"chunk_{i:03d}.wav" for i in range(len(manifest["chunks"]))]

    def find_wav(wav_name):
        # Look in this run's output dir first, then in the exported folder itself -
        # on Colab both are the same Drive path; on Kaggle the second one catches
        # .wav files uploaded back into the read-only dataset for cross-session resume.
        for candidate in (os.path.join(out_dir, wav_name),
                          os.path.join(patch_dir, "output", wav_name)):
            if os.path.exists(candidate):
                return candidate
        return None

    for chunk_filename in manifest["chunks"]:
        index = chunk_filename.split("_")[1].split(".")[0]  # chunk_000.txt -> 000
        wav_name = f"chunk_{index}.wav"
        if SKIP_EXISTING and find_wav(wav_name):
            print(f"skip {chunk_filename} (already synthesized)")
            continue

        with open(os.path.join(patch_dir, chunk_filename), "r", encoding="utf-8") as f:
            text = f.read()

        kwargs = {}
        if reference_wav_path:
            kwargs["reference_wav_path"] = reference_wav_path
            if prompt_text:
                kwargs["prompt_wav_path"] = reference_wav_path
                kwargs["prompt_text"] = prompt_text

        audio = model.generate(text=text, cfg_value=2.0, inference_timesteps=10, **kwargs)
        out_path = os.path.join(out_dir, wav_name)
        sf.write(out_path, audio, sample_rate)
        persist(out_path, entry["folder"] + "/output")
        print(f"wrote {out_path}")

    # Merge this patch right away (instead of one big merge at the end) so an
    # interrupted batch still yields finished result files for completed patches.
    missing = [w for w in manifest["expected_outputs"] if find_wav(w) is None]
    if missing:
        print(f"patch incomplete - {len(missing)} chunk(s) missing "
              f"(first: {missing[0]}); re-run this cell to resume")
        summary.append((label, f"incomplete ({len(missing)} chunks missing)"))
        continue

    parts = []
    merge_sr = None
    for wav_name in manifest["expected_outputs"]:
        audio, sr = sf.read(find_wav(wav_name))
        if merge_sr is None:
            merge_sr = sr
        parts.append(audio)
    sf.write(result_path, np.concatenate(parts), merge_sr)
    persist(result_path, "result")
    print(f"merged {len(parts)} chunks -> {result_path}")
    summary.append((label, "merged"))

print("\n=== Batch summary ===")
for name, status in summary:
    print(f"- {name}: {status}")

In [ ]:
# Cell 9: Kaggle only (per the global IS_KAGGLE flag) - zip the merged result/
# files so you can download them from the Output pane. In Kaggle Drive mode this
# is just a convenience copy (result/ is already uploaded to Drive); on Colab you
# don't need it at all.
import shutil

if IS_KAGGLE:
    archive = shutil.make_archive("/kaggle/working/results", "zip", RESULT_DIR)
    print("Download this from the Output pane:", archive)
else:
    print("Colab run - merged files are already saved in Drive:", RESULT_DIR)